# NjengaData — Data Cleaning Pipeline
### Notebook 01 | Data Engineer: Kimberly Wangui

This notebook is the foundation of the NjengaData pipeline.

**What happens here:**
- Raw data is loaded from three sources: KNBS, CAHF, and Kemika Inc
- Each dataset is inspected, cleaned, and normalised to a consistent format
- All cleaned data is loaded into a single SQLite database: `njenga.db`

**Output:** `data/njenga.db` — ready for Kelvin's analysis notebook.

> *"You cannot analyse dirty data. You cannot visualise wrong data.
> Clean data is not a step in the process — it is the foundation of everything."*

---
## Section 1: Imports and Configuration

Before we load any data, we import all the libraries we need and define
the file paths for our raw data sources and our output database.

This cell runs first — always.

In [1]:
# Standard library
import sqlite3

# Jupyter runs from the folder the notebook lives in (notebooks/)
# We need to move up one level to the project root (njenga-data/)
# so that all file paths resolve correctly on every team member's machine
import os

# Only move up if we are currently inside the notebooks/ folder
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

print("Working directory:", os.getcwd())

# Data manipulation
import pandas as pd

# Display settings — makes dataframes easier to read in the notebook
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)
pd.set_option('display.float_format', '{:,.2f}'.format)

# ── File paths ────────────────────────────────────────────────────────
# All paths are relative to the project root (njenga-data/)
# This means the notebook works on any team member's machine
# as long as they have cloned the repo correctly

RAW_KNBS    = "data/raw/knbs/material_price_index.csv"
RAW_CAHF    = "data/raw/cahf/cost_benchmarks.csv"
RAW_INCOME  = "data/raw/cahf/household_income.csv"
RAW_KEMIKA  = "data/raw/kemika/kemika_projects.csv"
DB_PATH     = "data/njenga.db"

print("Libraries loaded.")
print(f"Database will be written to: {DB_PATH}")

Working directory: /home/miringu/Documents/njenga-data
Libraries loaded.
Database will be written to: data/njenga.db


---
## Section 2: Load and Inspect Raw Data

We load each raw file and inspect it before touching anything.

**Why inspect first?**
Real government data is messy — wrong column names, inconsistent spelling,
missing values, wrong data types. We need to see the problems before we fix them.

We check four things for every dataset:
1. Shape — how many rows and columns?
2. Column names — are they what we expect?
3. Data types — are numbers stored as numbers, or as text?
4. Missing values — are there any nulls?

### 2.1 KNBS Material Price Index
*Source: Kenya National Bureau of Statistics — Construction Input Price Index*

In [2]:
# Load the KNBS material price index
df_knbs = pd.read_csv(RAW_KNBS)

# Inspect
print("Shape:", df_knbs.shape)
print("\nColumn names:", df_knbs.columns.tolist())
print("\nData types:\n", df_knbs.dtypes)
print("\nFirst 5 rows:")
df_knbs.head()

Shape: (72, 4)

Column names: ['year', 'quarter', 'material', 'index_value']

Data types:
 year             int64
quarter         object
material        object
index_value    float64
dtype: object

First 5 rows:


,year,quarter,material,index_value
0,2019,Q1,cement,100.00
1,2019,Q1,steel,100.00
2,2019,Q1,timber,100.00
3,2019,Q2,cement,101.20
4,2019,Q2,steel,102.10


### 2.2 CAHF Cost Benchmarks
*Source: Centre for Affordable Housing Finance Africa — Kenya Housing Development Cost Benchmark Report 2022*

In [3]:
# Load the CAHF cost benchmarks
df_cahf = pd.read_csv(RAW_CAHF)

print("Shape:", df_cahf.shape)
print("\nColumn names:", df_cahf.columns.tolist())
print("\nData types:\n", df_cahf.dtypes)
print("\nUnique counties:", df_cahf['county'].unique())
print("Unique unit types:", df_cahf['unit_type'].unique())
print("Unique cost categories:", df_cahf['cost_category'].unique())
print("\nFirst 5 rows:")
df_cahf.head()

Shape: (60, 5)

Column names: ['county', 'unit_type', 'cost_category', 'amount_kes', 'year']

Data types:
 county           object
unit_type        object
cost_category    object
amount_kes        int64
year              int64
dtype: object

Unique counties: ['Nairobi' 'Kiambu' 'Nakuru' 'Mombasa']
Unique unit types: ['2BR' '3BR' 'Studio']
Unique cost categories: ['Land' 'Materials' 'Labour' 'Compliance' 'Overheads']

First 5 rows:


,county,unit_type,cost_category,amount_kes,year
0,Nairobi,2BR,Land,850000,2022
1,Nairobi,2BR,Materials,1200000,2022
2,Nairobi,2BR,Labour,480000,2022
3,Nairobi,2BR,Compliance,420000,2022
4,Nairobi,2BR,Overheads,350000,2022


### 2.3 Household Income Data
*Source: World Bank / KNBS — Kenya Median Household Income by County 2022*

In [4]:
# Load household income data
df_income = pd.read_csv(RAW_INCOME)

print("Shape:", df_income.shape)
print("\nColumn names:", df_income.columns.tolist())
print("\nData types:\n", df_income.dtypes)
print("\nAll rows:")
df_income

Shape: (4, 3)

Column names: ['county', 'median_annual_income_kes', 'year']

Data types:
 county                      object
median_annual_income_kes     int64
year                         int64
dtype: object

All rows:


,county,median_annual_income_kes,year
0,Nairobi,720000,2022
1,Kiambu,540000,2022
2,Nakuru,420000,2022
3,Mombasa,480000,2022


### 2.4 Kemika Inc Project Records
*Source: Kemika Inc internal BOQ records — anonymised*

> Note: All client identifiers have been replaced with generic project IDs
> (e.g. Project_Nairobi_001) in compliance with our Data Privacy Policy.
> No real client names, plot numbers, or personal data appear in this dataset.

In [5]:
# Load Kemika Inc anonymised project records
df_kemika = pd.read_csv(RAW_KEMIKA)

print("Shape:", df_kemika.shape)
print("\nColumn names:", df_kemika.columns.tolist())
print("\nData types:\n", df_kemika.dtypes)
print("\nCounties covered:", df_kemika['county'].unique())
print("Years covered:", sorted(df_kemika['year'].unique()))
print("\nAll rows:")
df_kemika

Shape: (7, 9)

Column names: ['project_id', 'county', 'unit_type', 'land_kes', 'materials_kes', 'labour_kes', 'compliance_kes', 'overheads_kes', 'year']

Data types:
 project_id        object
county            object
unit_type         object
land_kes           int64
materials_kes      int64
labour_kes         int64
compliance_kes     int64
overheads_kes      int64
year               int64
dtype: object

Counties covered: ['Nairobi' 'Kiambu' 'Nakuru' 'Mombasa']
Years covered: [2023, 2024]

All rows:


,project_id,county,unit_type,land_kes,materials_kes,labour_kes,compliance_kes,overheads_kes,year
0,Project_Nairobi_001,Nairobi,3BR,1050000,1580000,640000,560000,460000,2023
1,Project_Nairobi_002,Nairobi,2BR,820000,1180000,465000,405000,338000,2023
2,Project_Nairobi_003,Nairobi,2BR,880000,1240000,490000,430000,360000,2024
3,Project_Kiambu_001,Kiambu,3BR,720000,1480000,590000,365000,348000,2023
4,Project_Kiambu_002,Kiambu,2BR,560000,1080000,428000,268000,270000,2024
5,Project_Nakuru_001,Nakuru,2BR,310000,960000,384000,190000,238000,2023
6,Project_Mombasa_001,Mombasa,3BR,780000,1410000,564000,250000,392000,2024


---
## Section 3: Data Cleaning and Normalisation

Each dataset is now cleaned and reshaped into a consistent format.

**The core problem:** CAHF and Kemika store cost data differently.
- CAHF uses **long format** — one row per cost category
- Kemika uses **wide format** — one row per project, costs as columns

We must melt Kemika into long format before combining the two sources.

**What melting means:** We take the five cost columns and collapse them
into two columns — `cost_category` and `amount_kes` — matching CAHF's structure.

### 3.1 Clean the KNBS Dataset

In [6]:
# Work on a copy — never modify the raw dataframe directly
# This means we can always go back to df_knbs if something goes wrong
knbs = df_knbs.copy()

# Standardise text columns — strip whitespace, fix capitalisation
# Real KNBS files often have trailing spaces or inconsistent casing
knbs['material'] = knbs['material'].str.strip().str.lower()
knbs['quarter']  = knbs['quarter'].str.strip().str.upper()

# Rename index_value to be more descriptive
knbs = knbs.rename(columns={'index_value': 'price_index'})

# Confirm the base year — 2019 Q1 should be 100.0 for all materials
# This is our sanity check against the source data
base_year_check = knbs[(knbs['year'] == 2019) & (knbs['quarter'] == 'Q1')]
print("Base year check (all should be 100.0):")
print(base_year_check[['material', 'price_index']].to_string(index=False))

print(f"\nKNBS clean shape: {knbs.shape}")
print("\nSample:")
knbs.head(6)

Base year check (all should be 100.0):
material  price_index
  cement       100.00
   steel       100.00
  timber       100.00

KNBS clean shape: (72, 4)

Sample:


,year,quarter,material,price_index
0,2019,Q1,cement,100.00
1,2019,Q1,steel,100.00
2,2019,Q1,timber,100.00
3,2019,Q2,cement,101.20
4,2019,Q2,steel,102.10
5,2019,Q2,timber,100.80


### 3.2 Clean the CAHF Dataset

In [7]:
# Work on a copy
cahf = df_cahf.copy()

# Standardise text columns
cahf['county']        = cahf['county'].str.strip()
cahf['unit_type']     = cahf['unit_type'].str.strip()
cahf['cost_category'] = cahf['cost_category'].str.strip()

# Confirm all five cost categories are present and correctly named
# These must match exactly what we use in analysis and charts
expected_categories = {'Land', 'Materials', 'Labour', 'Compliance', 'Overheads'}
actual_categories   = set(cahf['cost_category'].unique())
missing             = expected_categories - actual_categories
extra               = actual_categories - expected_categories

print("Expected categories:", expected_categories)
print("Actual categories:  ", actual_categories)
print("Missing:", missing if missing else "None")
print("Extra:  ", extra if extra else "None")

# Confirm all four counties are present
print("\nCounties:", sorted(cahf['county'].unique()))

print(f"\nCAHF clean shape: {cahf.shape}")
print("\nSample:")
cahf.head()

Expected categories: {'Compliance', 'Land', 'Overheads', 'Labour', 'Materials'}
Actual categories:   {'Land', 'Overheads', 'Labour', 'Compliance', 'Materials'}
Missing: None
Extra:   None

Counties: ['Kiambu', 'Mombasa', 'Nairobi', 'Nakuru']

CAHF clean shape: (60, 5)

Sample:


,county,unit_type,cost_category,amount_kes,year
0,Nairobi,2BR,Land,850000,2022
1,Nairobi,2BR,Materials,1200000,2022
2,Nairobi,2BR,Labour,480000,2022
3,Nairobi,2BR,Compliance,420000,2022
4,Nairobi,2BR,Overheads,350000,2022


### 3.3 Clean the Kemika Dataset and Melt to Long Format

Kemika data arrives in wide format — one row per project, costs as separate columns.
We melt it into long format to match CAHF's structure before combining the two sources.

In [8]:
# Work on a copy
kemika = df_kemika.copy()

# Standardise text columns
kemika['county']    = kemika['county'].str.strip()
kemika['unit_type'] = kemika['unit_type'].str.strip()

# These are the five cost columns we need to collapse
cost_columns = ['land_kes', 'materials_kes', 'labour_kes', 'compliance_kes', 'overheads_kes']

# Melt wide format into long format
# id_vars    — columns we keep as-is (they identify each row)
# value_vars — columns we collapse into two new columns
# var_name   — name for the new column that holds the old column names
# value_name — name for the new column that holds the values
kemika_long = kemika.melt(
    id_vars    = ['project_id', 'county', 'unit_type', 'year'],
    value_vars = cost_columns,
    var_name   = 'cost_category',
    value_name = 'amount_kes'
)

# The cost_category column now contains values like 'land_kes', 'materials_kes'
# We need to clean those up to match CAHF exactly: 'Land', 'Materials' etc.
category_map = {
    'land_kes'        : 'Land',
    'materials_kes'   : 'Materials',
    'labour_kes'      : 'Labour',
    'compliance_kes'  : 'Compliance',
    'overheads_kes'   : 'Overheads'
}
kemika_long['cost_category'] = kemika_long['cost_category'].map(category_map)

print(f"Kemika wide shape:  {kemika.shape}")
print(f"Kemika long shape:  {kemika_long.shape}")
print("\nCategories after mapping:", sorted(kemika_long['cost_category'].unique()))
print("\nSample — first 10 rows:")
kemika_long.head(10)

Kemika wide shape:  (7, 9)
Kemika long shape:  (35, 6)

Categories after mapping: ['Compliance', 'Labour', 'Land', 'Materials', 'Overheads']

Sample — first 10 rows:


,project_id,county,unit_type,year,cost_category,amount_kes
0,Project_Nairobi_001,Nairobi,3BR,2023,Land,1050000
1,Project_Nairobi_002,Nairobi,2BR,2023,Land,820000
2,Project_Nairobi_003,Nairobi,2BR,2024,Land,880000
3,Project_Kiambu_001,Kiambu,3BR,2023,Land,720000
4,Project_Kiambu_002,Kiambu,2BR,2024,Land,560000
5,Project_Nakuru_001,Nakuru,2BR,2023,Land,310000
6,Project_Mombasa_001,Mombasa,3BR,2024,Land,780000
7,Project_Nairobi_001,Nairobi,3BR,2023,Materials,1580000
8,Project_Nairobi_002,Nairobi,2BR,2023,Materials,1180000
9,Project_Nairobi_003,Nairobi,2BR,2024,Materials,1240000


### 3.4 Clean the Household Income Dataset

In [9]:
# Work on a copy
income = df_income.copy()

# Standardise text
income['county'] = income['county'].str.strip()

# Confirm all four counties are present
print("Counties:", sorted(income['county'].unique()))

# Add a monthly income column — useful for the affordability gap narrative
# Annual income divided by 12
income['median_monthly_income_kes'] = (income['median_annual_income_kes'] / 12).astype(int)

print(f"\nIncome clean shape: {income.shape}")
print("\nAll rows:")
income

Counties: ['Kiambu', 'Mombasa', 'Nairobi', 'Nakuru']

Income clean shape: (4, 4)

All rows:


,county,median_annual_income_kes,year,median_monthly_income_kes
0,Nairobi,720000,2022,60000
1,Kiambu,540000,2022,45000
2,Nakuru,420000,2022,35000
3,Mombasa,480000,2022,40000


---
## Section 4: Combining CAHF and Kemika Data

Both datasets now share the same structure — long format with matching column names.
We combine them into a single master cost dataframe.

CAHF provides county-level benchmarks from 2022 research.
Kemika provides real project records from 2023 and 2024.
Together they give us both the benchmark and the ground truth.

In [10]:
# Add a source column to each dataset before combining
# This lets us filter by source later in analysis
cahf['source']        = 'CAHF'
kemika_long['source'] = 'Kemika'

# CAHF has county, unit_type, cost_category, amount_kes, year, source
# Kemika long has project_id, county, unit_type, year, cost_category, amount_kes, source
# We only keep the columns that exist in both

shared_columns = ['county', 'unit_type', 'cost_category', 'amount_kes', 'year', 'source']

cahf_trimmed   = cahf[shared_columns]
kemika_trimmed = kemika_long[shared_columns]

# Stack them vertically — pd.concat joins dataframes top to bottom
df_costs = pd.concat([cahf_trimmed, kemika_trimmed], ignore_index=True)

print(f"CAHF rows:   {len(cahf_trimmed)}")
print(f"Kemika rows: {len(kemika_trimmed)}")
print(f"Combined:    {len(df_costs)}")
print(f"\nExpected total: {len(cahf_trimmed) + len(kemika_trimmed)}")
print(f"Match: {len(df_costs) == len(cahf_trimmed) + len(kemika_trimmed)}")

print("\nSources in combined dataset:", df_costs['source'].unique())
print("\nSample:")
df_costs.head(8)

CAHF rows:   60
Kemika rows: 35
Combined:    95

Expected total: 95
Match: True

Sources in combined dataset: ['CAHF' 'Kemika']

Sample:


,county,unit_type,cost_category,amount_kes,year,source
0,Nairobi,2BR,Land,850000,2022,CAHF
1,Nairobi,2BR,Materials,1200000,2022,CAHF
2,Nairobi,2BR,Labour,480000,2022,CAHF
3,Nairobi,2BR,Compliance,420000,2022,CAHF
4,Nairobi,2BR,Overheads,350000,2022,CAHF
5,Nairobi,3BR,Land,1100000,2022,CAHF
6,Nairobi,3BR,Materials,1650000,2022,CAHF
7,Nairobi,3BR,Labour,660000,2022,CAHF


---
## Section 5: Loading Data into SQLite

We now write all cleaned dataframes into a single SQLite database: `njenga.db`

**Why SQLite?**
- It is a single file — easy to commit to GitHub and share with the team
- Kelvin queries it with standard SQL — no server, no configuration needed
- It is the bridge between the cleaning pipeline and the analysis layer

**Tables we create:**
| Table | Source | Purpose |
|---|---|---|
| `material_prices` | KNBS | Material price trend analysis |
| `cost_benchmarks` | CAHF + Kemika | Cost breakdown and county comparison |
| `household_income` | World Bank / KNBS | Affordability gap chart |

In [11]:
# Create a connection to the SQLite database
# If njenga.db does not exist, SQLite creates it automatically
conn = sqlite3.connect(DB_PATH)

print(f"Connected to: {DB_PATH}")
print(f"Database file exists: {os.path.exists(DB_PATH)}")

Connected to: data/njenga.db
Database file exists: True


### 5.1 Write the Tables

`if_exists='replace'` means if the table already exists, drop it and rebuild.
This ensures the database is always in sync with the cleaning pipeline.
Running this notebook twice produces the same result — no duplicates.
This property is called **idempotency** — an important concept in data engineering.

In [12]:
# Write each cleaned dataframe to its own table in njenga.db
# index=False means we do not write the Pandas row index as a column

knbs.to_sql('material_prices',  conn, if_exists='replace', index=False)
df_costs.to_sql('cost_benchmarks', conn, if_exists='replace', index=False)
income.to_sql('household_income',  conn, if_exists='replace', index=False)

print("Tables written successfully.")
print("\nTables in njenga.db:")

# Verify by querying the SQLite master table
# sqlite_master is SQLite's internal directory of all tables
tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", conn)
print(tables.to_string(index=False))

Tables written successfully.

Tables in njenga.db:
            name
 material_prices
 cost_benchmarks
household_income


### 5.2 Verify Table Contents

Before handing off to the Analyst, we run a quick row count on each table.
This is the engineer's final quality check — confirm the numbers match expectations.

In [13]:
# Row count verification for each table
verification_queries = {
    'material_prices' : "SELECT COUNT(*) as row_count FROM material_prices",
    'cost_benchmarks' : "SELECT COUNT(*) as row_count FROM cost_benchmarks",
    'household_income': "SELECT COUNT(*) as row_count FROM household_income"
}

print("Table row counts:")
print("-" * 35)

for table_name, query in verification_queries.items():
    result = pd.read_sql(query, conn)
    count  = result['row_count'].iloc[0]
    print(f"  {table_name:<20} {count:>5} rows")

print("-" * 35)
print("\nExpected:")
print(f"  {'material_prices':<20} {'72':>5} rows  (3 materials × 4 quarters × 6 years)")
print(f"  {'cost_benchmarks':<20} {'95':>5} rows  (60 CAHF + 35 Kemika)")
print(f"  {'household_income':<20} {'4':>5} rows  (4 counties)")

Table row counts:
-----------------------------------
  material_prices         72 rows
  cost_benchmarks         95 rows
  household_income         4 rows
-----------------------------------

Expected:
  material_prices         72 rows  (3 materials × 4 quarters × 6 years)
  cost_benchmarks         95 rows  (60 CAHF + 35 Kemika)
  household_income         4 rows  (4 counties)


### 5.3 Spot Check — Sample Rows from Each Table
A final human eye check before closing the connection.

In [14]:
# Pull 3 sample rows from each table and display them
for table in ['material_prices', 'cost_benchmarks', 'household_income']:
    print(f"\n--- {table} ---")
    sample = pd.read_sql(f"SELECT * FROM {table} LIMIT 3", conn)
    print(sample.to_string(index=False))


--- material_prices ---
 year quarter material  price_index
 2019      Q1   cement       100.00
 2019      Q1    steel       100.00
 2019      Q1   timber       100.00

--- cost_benchmarks ---
 county unit_type cost_category  amount_kes  year source
Nairobi       2BR          Land      850000  2022   CAHF
Nairobi       2BR     Materials     1200000  2022   CAHF
Nairobi       2BR        Labour      480000  2022   CAHF

--- household_income ---
 county  median_annual_income_kes  year  median_monthly_income_kes
Nairobi                    720000  2022                      60000
 Kiambu                    540000  2022                      45000
 Nakuru                    420000  2022                      35000


In [15]:
# Close the database connection cleanly
# Always close the connection when you are done writing
# Leaving it open can cause locking issues when Kelvin opens the same file
conn.close()

print("Connection closed.")
print(f"\nnjenga.db is ready for analysis.")
print(f"File size: {os.path.getsize(DB_PATH):,} bytes")
print("\nHandoff to Kelvin — open 02_analysis.ipynb")

Connection closed.

njenga.db is ready for analysis.
File size: 16,384 bytes

Handoff to Kelvin — open 02_analysis.ipynb


---
## Notebook Complete — Data Engineering Handoff

**Engineer:** Kimberly Wangui
**Status:** Complete
**Output:** `data/njenga.db`

### What was built
| Step | Action | Output |
|---|---|---|
| 1 | Loaded KNBS, CAHF, Kemika, Income data | 4 raw dataframes |
| 2 | Cleaned and standardised all sources | Consistent formats |
| 3 | Melted Kemika from wide to long format | 35 rows from 7 projects |
| 4 | Combined CAHF and Kemika cost data | 95 rows, single source |
| 5 | Loaded all tables into SQLite | njenga.db — 3 tables |

### Handoff checklist
- [ ] `njenga.db` confirmed present in `data/` folder
- [ ] All three tables verified with correct row counts
- [ ] No client names or personal data in Kemika records
- [ ] Notebook runs clean from Restart & Run All
- [ ] Committed to `dev-ingestion` branch on GitHub